In [1]:
# Jupyter Notebook UI for Attendance System

# Run this cell first to install requirements
!pip install ipywidgets opencv-python-headless face-recognition pandas openpyxl pillow imutils dlib

# Import required libraries
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import subprocess
import sys
import os

# Create UI elements
output = widgets.Output()
status_output = widgets.Output()

# Title
title = widgets.HTML(
    value="<h1 style='text-align: center; color: #2c3e50;'>Smart Face Recognition Attendance System</h1>"
)

subtitle = widgets.HTML(
    value="<h3 style='text-align: center; color: #34495e;'>With Anti-Spoofing (Blink Detection)</h3>"
)

# Buttons
register_btn = widgets.Button(
    description='📝 Register Student',
    button_style='primary',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

attendance_btn = widgets.Button(
    description='📸 Take Attendance',
    button_style='success',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

train_btn = widgets.Button(
    description='🔄 Train Encodings',
    button_style='warning',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

exit_btn = widgets.Button(
    description='🚪 Exit',
    button_style='danger',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

# Info box
info_box = widgets.HTML(
    value="""
    <div style='background-color: #ecf0f1; padding: 15px; border-radius: 5px; margin-top: 20px;'>
        <h4 style='color: #2c3e50;'>System Information:</h4>
        <ul>
            <li>📁 Dataset folder: dataset/</li>
            <li>📊 Attendance file: attendance/attendance.xlsx</li>
            <li>🔍 Liveness detection: Blink-based</li>
            <li>👥 Multiple face support: Yes</li>
        </ul>
    </div>
    """
)

# Registration form
name_input = widgets.Text(
    placeholder='Enter student name',
    description='Name:',
    layout=widgets.Layout(width='400px')
)

roll_input = widgets.Text(
    placeholder='Enter roll number',
    description='Roll No:',
    layout=widgets.Layout(width='400px')
)

class_input = widgets.Text(
    placeholder='Enter class',
    description='Class:',
    layout=widgets.Layout(width='400px')
)

register_form = widgets.VBox([
    widgets.HTML("<h4 style='color: #2c3e50;'>Registration Form</h4>"),
    name_input,
    roll_input,
    class_input,
    widgets.HTML("<p style='color: #7f8c8d;'>After submitting, camera will open for face capture</p>")
])

# Button click handlers
def on_register_clicked(b):
    with output:
        clear_output()
        display(register_form)

        # Create submit button
        submit_btn = widgets.Button(
            description='Start Registration',
            button_style='success'
        )

        def on_submit(b):
            with status_output:
                clear_output()

                # Get values
                name = name_input.value
                roll = roll_input.value
                student_class = class_input.value

                if not name or not roll or not student_class:
                    print("⚠️ Please fill all fields")
                    return

                print(f"📝 Registering: {name} (Roll: {roll})")
                print("Opening camera for face capture...")

                # Call registration script
                import subprocess
                import sys

                # Save registration info to temp file
                with open('temp_reg.txt', 'w') as f:
                    f.write(f"{name}\n{roll}\n{student_class}")

                # Run registration in separate process
                subprocess.Popen([sys.executable, '-c', '''
import pickle
import sys
sys.path.append('.')
from register import StudentRegistration

# Read registration data
with open('temp_reg.txt', 'r') as f:
    name = f.readline().strip()
    roll = f.readline().strip()
    student_class = f.readline().strip()

# Register student
reg = StudentRegistration()
valid, msg = reg.validate_input(name, roll, student_class)
if valid:
    success, msg = reg.capture_faces(name, roll, student_class)
    if success:
        success, msg = reg.generate_encodings(name, roll, student_class)
        print(f"✓ Registration complete: {msg}")
    else:
        print(f"✗ Error: {msg}")
else:
    print(f"✗ Error: {msg}")

# Clean up
import os
os.remove('temp_reg.txt')
input("\\nPress Enter to close...")
'''])

        submit_btn.on_click(on_submit)
        display(submit_btn)
        display(status_output)

def on_attendance_clicked(b):
    with output:
        clear_output()
        print("📸 Starting Attendance System...")
        print("Camera will open in a new window")
        print("Press ESC in camera window to stop")

        # Run attendance in separate process
        subprocess.Popen([sys.executable, '-c', '''
import sys
sys.path.append('.')
from attendance import AttendanceMarker
print("\\n🔍 Look at camera and blink naturally")
print("Press ESC to stop\\n")
attendance = AttendanceMarker()
attendance.run_attendance()
print("\\n✓ Attendance session completed")
input("\\nPress Enter to close...")
'''])

def on_train_clicked(b):
    with output:
        clear_output()
        print("🔄 Training face encodings from dataset...")

        from trainer import train_from_dataset
        train_from_dataset()

        print("\n✓ Training complete!")

def on_exit_clicked(b):
    with output:
        clear_output()
        print("👋 Thank you for using the system!")
        # Optionally close the notebook
        import sys
        sys.exit()

# Attach handlers
register_btn.on_click(on_register_clicked)
attendance_btn.on_click(on_attendance_clicked)
train_btn.on_click(on_train_clicked)
exit_btn.on_click(on_exit_clicked)

# Layout
button_box = widgets.VBox([
    register_btn,
    attendance_btn,
    train_btn,
    exit_btn
], layout=widgets.Layout(align_items='center'))

main_layout = widgets.VBox([
    title,
    subtitle,
    widgets.HTML("<hr>"),
    button_box,
    info_box,
    output
])

# Display UI
display(main_layout)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 6.3 MB/s eta 0:00:00m eta 0:00:010:01:01

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
